# UTS Data Mining II - Text
Alur: baca data → EDA → cleaning → tokenisasi → stopword → TF-IDF → K-Means → visualisasi.

Ganti lokasi CSV dan nama kolom `content` sesuai data soal. Contoh path memakai Colab.
Di Windows bisa memakai `r"D:\folder\data.csv"`. Jalankan cell dari atas ke bawah.
Perbandingan NLTK dan Sastrawi mengikuti notebook contoh; bagian Sastrawi di sini hanya stopword.

In [ ]:
# Jalankan sekali jika package atau data NLTK belum tersedia:
# %pip install pandas matplotlib scikit-learn nltk Sastrawi wordcloud openpyxl
# import nltk
# nltk.download("punkt_tab")
# nltk.download("stopwords")

## Baca data

In [ ]:
import pandas as pd

df = pd.read_csv("/content/BSI-reviews.csv")
display(df.head())

## EDA

In [ ]:
df.info()
display(df.isnull().sum())
display(df["content"].describe())

df = df.dropna(subset=["content"]).copy()
df["content"] = df["content"].astype(str)
df["content_length"] = df["content"].str.len()
display(df["content_length"].describe())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(df["content_length"], bins=30)
plt.xlabel("Panjang teks")
plt.ylabel("Jumlah review")
plt.title("Distribusi panjang review")
plt.show()

In [ ]:
from wordcloud import WordCloud

all_reviews = " ".join(df["content"])
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(all_reviews)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

## Cleaning

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(text.split())

df["cleaned_content"] = df["content"].apply(clean_text)
df = df[df["cleaned_content"].str.strip().ne("")].reset_index(drop=True)
display(df[["content", "cleaned_content"]].head())

## Tokenisasi

In [ ]:
from nltk.tokenize import word_tokenize

df["tokens"] = df["cleaned_content"].apply(word_tokenize)
display(df[["cleaned_content", "tokens"]].head())

## Stopword NLTK

In [ ]:
from nltk.corpus import stopwords

stop_words_nltk = set(stopwords.words("indonesian"))
df["tokens_nltk"] = df["tokens"].apply(lambda tokens: [word for word in tokens if word not in stop_words_nltk])
df["text_nltk"] = df["tokens_nltk"].apply(" ".join)
display(df[["tokens", "tokens_nltk"]].head())

## Stopword Sastrawi

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stop_words_sastrawi = set(StopWordRemoverFactory().get_stop_words())
df["tokens_sastrawi"] = df["tokens"].apply(lambda tokens: [word for word in tokens if word not in stop_words_sastrawi])
df["text_sastrawi"] = df["tokens_sastrawi"].apply(" ".join)
display(df[["tokens", "tokens_sastrawi"]].head())

## TF-IDF
Pakai baris yang masih memiliki kata pada kedua metode agar data perbandingannya sama.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df = df[df["text_nltk"].ne("") & df["text_sastrawi"].ne("")].reset_index(drop=True)

vectorizer_nltk = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
tfidf_nltk = vectorizer_nltk.fit_transform(df["text_nltk"])

vectorizer_sastrawi = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
tfidf_sastrawi = vectorizer_sastrawi.fit_transform(df["text_sastrawi"])

print("Shape TF-IDF NLTK:", tfidf_nltk.shape)
print("Shape TF-IDF Sastrawi:", tfidf_sastrawi.shape)

## Modeling dan evaluasi NLTK
Sesuaikan daftar k dengan soal. Setiap k harus lebih kecil dari jumlah data dan membentuk minimal dua cluster.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_list = [3, 5, 7]
scores_nltk = []

for k in k_list:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(tfidf_nltk)
    score = silhouette_score(tfidf_nltk, labels)
    scores_nltk.append(score)
    print(f"K={k}, silhouette NLTK={score:.4f}")

## Modeling dan evaluasi Sastrawi

In [ ]:
scores_sastrawi = []

for k in k_list:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(tfidf_sastrawi)
    score = silhouette_score(tfidf_sastrawi, labels)
    scores_sastrawi.append(score)
    print(f"K={k}, silhouette Sastrawi={score:.4f}")

df_scores = pd.DataFrame({"k": k_list, "NLTK": scores_nltk, "Sastrawi": scores_sastrawi})
display(df_scores)

In [ ]:
plt.plot(k_list, scores_nltk, marker="o", label="NLTK")
plt.plot(k_list, scores_sastrawi, marker="o", label="Sastrawi")
plt.xlabel("K")
plt.ylabel("Silhouette score")
plt.title("Perbandingan NLTK dan Sastrawi")
plt.legend()
plt.show()

## Model akhir
Ganti `k = 7` sesuai hasil evaluasi atau ketentuan soal.

In [ ]:
k = 7
kmeans_nltk = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster_nltk"] = kmeans_nltk.fit_predict(tfidf_nltk)

kmeans_sastrawi = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster_sastrawi"] = kmeans_sastrawi.fit_predict(tfidf_sastrawi)
display(df[["content", "cluster_nltk", "cluster_sastrawi"]].head())

## Visualisasi cluster
SVD hanya untuk grafik. K-Means dan silhouette memakai TF-IDF sebelum reduksi.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd_nltk = TruncatedSVD(n_components=2, random_state=42)
plot_nltk = svd_nltk.fit_transform(tfidf_nltk)

svd_sastrawi = TruncatedSVD(n_components=2, random_state=42)
plot_sastrawi = svd_sastrawi.fit_transform(tfidf_sastrawi)

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.scatter(plot_nltk[:, 0], plot_nltk[:, 1], c=df["cluster_nltk"], cmap="viridis", alpha=0.6)
plt.title("Cluster NLTK")
plt.xlabel("Komponen SVD 1")
plt.ylabel("Komponen SVD 2")

plt.subplot(1, 2, 2)
plt.scatter(plot_sastrawi[:, 0], plot_sastrawi[:, 1], c=df["cluster_sastrawi"], cmap="viridis", alpha=0.6)
plt.title("Cluster Sastrawi")
plt.xlabel("Komponen SVD 1")
plt.ylabel("Komponen SVD 2")
plt.tight_layout()
plt.show()

## Kata utama dan contoh teks - NLTK

In [ ]:
terms_nltk = vectorizer_nltk.get_feature_names_out()

for cluster in range(k):
    top_indices = kmeans_nltk.cluster_centers_[cluster].argsort()[-10:][::-1]
    print(f"Cluster {cluster}:", ", ".join(terms_nltk[top_indices]))
    display(df.loc[df["cluster_nltk"] == cluster, ["content"]].head(3))

## Kata utama dan contoh teks - Sastrawi

In [ ]:
terms_sastrawi = vectorizer_sastrawi.get_feature_names_out()

for cluster in range(k):
    top_indices = kmeans_sastrawi.cluster_centers_[cluster].argsort()[-10:][::-1]
    print(f"Cluster {cluster}:", ", ".join(terms_sastrawi[top_indices]))
    display(df.loc[df["cluster_sastrawi"] == cluster, ["content"]].head(3))

## Simpan hasil (opsional)
Hapus tanda `#` untuk menyimpan CSV.

In [ ]:
# df.to_csv("text_hasil.csv", index=False)
# df_scores.to_csv("text_skor_k.csv", index=False)

## Alternatif input
Ganti cell **Baca data** dengan salah satu contoh berikut. Setelahnya tetap pakai nama kolom `content`, atau ganti semua `content` sesuai kolom soal.

**Excel**
```python
import pandas as pd
df = pd.read_excel("/content/text.xlsx")
display(df.head())
```

**Kaggle berisi CSV** — install `kagglehub` terlebih dahulu, lalu ganti slug dan nama file.
```python
import pandas as pd
import kagglehub
from pathlib import Path

folder = Path(kagglehub.dataset_download("pemilik/nama-dataset"))
print(list(folder.iterdir()))
df = pd.read_csv(folder / "data.csv")
display(df.head())
```
Untuk Excel dari Kaggle, cukup ganti `pd.read_csv(...)` menjadi `pd.read_excel(...)`.

## Stemming (kalau diminta soal)
Tambahkan sebelum TF-IDF. Jalankan ulang TF-IDF sampai akhir setelah stemming.
```python
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
stemmer = StemmerFactory().create_stemmer()
df["text_sastrawi"] = df["text_sastrawi"].apply(stemmer.stem)
```
Silhouette lebih tinggi berarti pemisahan lebih baik pada fitur yang dipakai. Baca juga contoh teks sebelum memberi nama topik cluster.